# 01 — Raw data quality check (Open-Meteo only)

**Purpose:** establish whether the active Karak datasets are complete and internally consistent before feature engineering.

**Order:** run this after `python -m src.ingest`, before notebook 02. The notebook selects only files with the explicit `karak_aqi_training_open_meteo_hourly_...` and `karak_weather_features_open_meteo_hourly_...` names. No secondary-provider file can satisfy these patterns.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Raw data directory:', RAW_DIR)


Project root: E:\10pearls-internship-2026\development
Raw data directory: E:\10pearls-internship-2026\development\data\raw


In [2]:
def newest(pattern):
    matches = sorted(RAW_DIR.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'No file matched {pattern}. Run `python -m src.ingest` first.')
    return matches[-1]

aq_path = newest('karak_aqi_training_open_meteo_hourly_*.csv')
wx_path = newest('karak_weather_features_open_meteo_hourly_*.csv')
aq = pd.read_csv(aq_path, parse_dates=['time'])
wx = pd.read_csv(wx_path, parse_dates=['time'])
print('AQ file:', aq_path.name)
print('Weather file:', wx_path.name)
print('AQ rows/columns:', len(aq), len(aq.columns))
print('Weather rows/columns:', len(wx), len(wx.columns))
print('AQ range:', aq['time'].min(), '->', aq['time'].max())
print('Weather range:', wx['time'].min(), '->', wx['time'].max())
print('AQ source labels:', aq['source'].dropna().unique().tolist())
print('Weather source labels:', wx['source'].dropna().unique().tolist())
expected_aq_start = pd.Timestamp('2022-08-05')
# Derive the requested end date from the filename and require both files to
# contain that same final hour. This remains valid on a future data refresh.
def filename_end_date(path):
    return path.name.split('_to_', 1)[1].split('_', 1)[0]
file_end_aq = filename_end_date(aq_path)
file_end_wx = filename_end_date(wx_path)
actual_end_date = aq['time'].max().date().isoformat()
contract_pass = (
    aq_path.name.startswith('karak_aqi_training_open_meteo_hourly_2022-08-05_to_')
    and wx_path.name.startswith('karak_weather_features_open_meteo_hourly_2022-08-05_to_')
    and aq['time'].min() == expected_aq_start
    and wx['time'].min() == expected_aq_start
    and aq['time'].max() == wx['time'].max()
    and file_end_aq == file_end_wx == actual_end_date
    and all(str(label).startswith('open_meteo_') for label in aq['source'].dropna().unique())
    and all(str(label).startswith('open_meteo_') for label in wx['source'].dropna().unique())
)
print('Filename/date/source contract:', 'PASS' if contract_pass else 'REVIEW')


AQ file: karak_aqi_training_open_meteo_hourly_2022-08-05_to_2026-07-31_20260731_181040.csv
Weather file: karak_weather_features_open_meteo_hourly_2022-08-05_to_2026-07-31_20260731_181049.csv
AQ rows/columns: 34968 11
Weather rows/columns: 34968 12
AQ range: 2022-08-05 00:00:00 -> 2026-07-31 23:00:00
Weather range: 2022-08-05 00:00:00 -> 2026-07-31 23:00:00
AQ source labels: ['open_meteo_air_quality']
Weather source labels: ['open_meteo_weather_features']
Filename/date/source contract: PASS


### Finding from file identity and coverage

The output immediately above is the audit trail: it records the exact filenames, row counts, date ranges, and source labels used for this run. The active files should report only `open_meteo_*` labels. The following cell converts those values into a pass/fail statement so a later refresh cannot silently reuse the old narrative.

**Diagnostic note from the first pull:** the original 2022-08-01 start produced a contiguous 77-hour all-variable AQ null block through 2022-08-04 04:00. It was not imputed. The active AQ and weather-feature files begin on 2022-08-05; the original pulls are retained outside `data/raw/` for audit only.


In [3]:
def hourly_gap_report(frame, label):
    times = pd.to_datetime(frame['time'])
    duplicates = int(times.duplicated().sum())
    expected = pd.date_range(times.min(), times.max(), freq='h')
    missing = expected.difference(pd.DatetimeIndex(times))
    print(f'{label}: duplicates={duplicates}, missing hourly timestamps={len(missing)}, expected_hours={len(expected)}')
    return duplicates, missing

aq_dupes, aq_missing = hourly_gap_report(aq, 'AQ')
wx_dupes, wx_missing = hourly_gap_report(wx, 'weather')
na_report = pd.DataFrame({'aq_missing_values': aq.isna().sum(), 'weather_missing_values': wx.isna().sum()}).fillna(0).astype(int)
print('\nMissing-value counts by column:')
print(na_report.to_string())
quality_pass = (contract_pass and aq_dupes == 0 and wx_dupes == 0 and len(aq_missing) == 0 and len(wx_missing) == 0 and int(aq.isna().sum().sum()) == 0 and int(wx.isna().sum().sum()) == 0)
print('\nQC verdict:', 'PASS — no duplicates, hourly gaps, or missing cells in the active files.' if quality_pass else 'REVIEW — at least one completeness rule failed; do not train yet.')


AQ: duplicates=0, missing hourly timestamps=0, expected_hours=34968
weather: duplicates=0, missing hourly timestamps=0, expected_hours=34968

Missing-value counts by column:
                       aq_missing_values  weather_missing_values
aerosol_optical_depth                  0                       0
carbon_monoxide                        0                       0
cloud_cover                            0                       0
dew_point_2m                           0                       0
dust                                   0                       0
nitrogen_dioxide                       0                       0
ozone                                  0                       0
pm10                                   0                       0
pm2_5                                  0                       0
precipitation                          0                       0
rain                                   0                       0
relative_humidity_2m                   0      

### Findings from the executed completeness check

The printed `QC verdict` is the authoritative result for this pull. A `PASS` means the two active Open-Meteo files can proceed to notebook 02 without imputation for missing cells or timestamps. A `REVIEW` means the files must remain active but the issue must be fixed and this notebook rerun before modeling. The initial 77-hour source gap was resolved by changing the requested start date, not by filling values.


In [4]:
aq_numeric = aq.select_dtypes(include='number')
negative_counts = (aq_numeric < 0).sum().sort_values(ascending=False)
print('Negative-value counts in AQ numeric columns:')
print(negative_counts.to_string())
for column in ['pm2_5', 'pm10', 'ozone']:
    if column in aq:
        print(f'{column}: min={aq[column].min():.3f}, median={aq[column].median():.3f}, max={aq[column].max():.3f}')
print('\nSource labels:', aq['source'].dropna().unique().tolist(), wx['source'].dropna().unique().tolist())


Negative-value counts in AQ numeric columns:
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
pm2_5: min=0.100, median=25.500, max=129.000
pm10: min=0.100, median=39.000, max=536.400
ozone: min=0.000, median=100.000, max=204.000

Source labels: ['open_meteo_air_quality'] ['open_meteo_weather_features']


### Finding from physical-range checks

The numeric output above is deliberately retained with the notebook. Negative pollutant concentrations indicate a source/parse problem; unusually large values are flagged for review but are not automatically deleted because dust events can be real. Source labels must identify only `open_meteo_*` values in the active workflow.


In [5]:
summary = pd.DataFrame({
    'rows': [len(aq), len(wx)],
    'columns': [len(aq.columns), len(wx.columns)],
    'start': [aq.time.min(), wx.time.min()],
    'end': [aq.time.max(), wx.time.max()],
    'missing_cells': [int(aq.isna().sum().sum()), int(wx.isna().sum().sum())],
    'contract_pass': [contract_pass, contract_pass],
}, index=['aqi_training', 'weather_features'])
summary.to_csv(PROCESSED_DIR / 'qc_raw_open_meteo_summary.csv')
print(summary.to_string())
print('\nSaved evidence table:', PROCESSED_DIR / 'qc_raw_open_meteo_summary.csv')


                   rows  columns      start                 end  missing_cells  contract_pass
aqi_training      34968       11 2022-08-05 2026-07-31 23:00:00              0           True
weather_features  34968       12 2022-08-05 2026-07-31 23:00:00              0           True

Saved evidence table: E:\10pearls-internship-2026\development\data\processed\qc_raw_open_meteo_summary.csv


## 01 decision

This notebook does not compare providers. It verifies that the **single active provider's two products** are structurally usable. The historical OpenWeather/AQICN sanity check is documented in `data_sources_and_file_naming.md` and is not required to pass this notebook.
